# Sustainable Product Lifecycle Management Platform
## Real-Time Recyclability Score Calculation: Scientific Validation & Sensitivity Analysis

**Author:** Mani (5th Semester Biotechnology)  
**Objective:** Independently validate the Spring Boot recyclability scoring algorithm in Python (Pandas/NumPy), measure **Band Agreement %** against a 22-product benchmark, evaluate 50/50 calibration, and perform $\pm 10\%$ parameter sensitivity testing.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split

# 1. Load Data
materials_df = pd.read_csv('materials_reference.csv').set_index('name')
benchmark_df = pd.read_csv('benchmark_products.csv')
print(f'Loaded {len(materials_df)} reference materials and {len(benchmark_df)} benchmark products.')
materials_df.head()

### 2. Scoring Algorithm Implementation
Mathematical parity with `ScoringService.java`:

In [ ]:
DEFAULT_WEIGHTS = {
    'recyclability': 0.25,
    'recycled_content': 0.15,
    'disassembly': 0.15,
    'reusability': 0.15,
    'repairability': 0.15,
    'lifetime': 0.15,
    'hazardous_penalty': 0.30,
    'max_lifetime_years': 20.0,
    'threshold_poor': 40.0,
    'threshold_moderate': 70.0
}

def calculate_score(composition_json, weights=DEFAULT_WEIGHTS):
    items = json.loads(composition_json)
    total_weight = sum(item['weight'] for item in items)
    if total_weight <= 0:
        return 0.0, 'Poor'

    hazardous_weight = 0.0
    sum_rec = 0.0
    sum_rc = 0.0
    sum_dis = 0.0
    sum_reu = 0.0
    sum_rep = 0.0
    sum_life = 0.0

    for item in items:
        mat_name = item['material']
        w = item['weight']
        mat = materials_df.loc[mat_name]

        if mat['hazardous']:
            hazardous_weight += w

        sum_rec += mat['recyclability_pct'] * w
        sum_rc += mat['recycled_content_pct'] * w
        sum_dis += (mat['disassembly_ease'] / 10.0 * 100.0) * w
        sum_reu += mat['reusability_pct'] * w
        sum_rep += (mat['repairability'] / 10.0 * 100.0) * w
        sum_life += min(mat['lifetime_years'] / weights['max_lifetime_years'], 1.0) * 100.0 * w

    positive = (
        weights['recyclability'] * (sum_rec / total_weight) +
        weights['recycled_content'] * (sum_rc / total_weight) +
        weights['disassembly'] * (sum_dis / total_weight) +
        weights['reusability'] * (sum_reu / total_weight) +
        weights['repairability'] * (sum_rep / total_weight) +
        weights['lifetime'] * (sum_life / total_weight)
    )

    hazardous_share = (hazardous_weight / total_weight) * 100.0
    hazard_penalty = weights['hazardous_penalty'] * hazardous_share
    final_score = round(max(0.0, min(100.0, positive - hazard_penalty)), 1)

    if final_score <= weights['threshold_poor']:
        label = 'Poor'
    elif final_score <= weights['threshold_moderate']:
        label = 'Moderate'
    else:
        label = 'Good'

    return final_score, label

### 3. Benchmark Accuracy & Band Agreement Evaluation

In [ ]:
scores, bands = zip(*[calculate_score(row['composition_json']) for _, row in benchmark_df.iterrows()])
benchmark_df['calculated_score'] = scores
benchmark_df['calculated_band'] = bands
benchmark_df['agreement'] = benchmark_df['calculated_band'] == benchmark_df['expected_band']

agreement_pct = benchmark_df['agreement'].mean() * 100.0
print(f'Measured Band Agreement: {agreement_pct:.1f}% ({benchmark_df["agreement"].sum()}/{len(benchmark_df)})')
benchmark_df[['product_name', 'total_weight_g', 'calculated_score', 'calculated_band', 'expected_band', 'agreement']].head(10)

### 4. 50/50 Calibration Split Validation

In [ ]:
train_df, test_df = train_test_split(benchmark_df, test_size=0.5, random_state=42, stratify=benchmark_df['expected_band'])
print(f'Calibration Agreement (50% train): {(train_df["agreement"].mean() * 100):.1f}%')
print(f'Generalization Agreement (50% test): {(test_df["agreement"].mean() * 100):.1f}%')

### 5. Sensitivity Analysis ($\pm 10\%$ Perturbations)

In [ ]:
base_scores = benchmark_df['calculated_score'].values
keys = ['recyclability', 'recycled_content', 'disassembly', 'reusability', 'repairability', 'lifetime']
results = {}

for k in keys:
    w_mod = DEFAULT_WEIGHTS.copy()
    w_mod[k] *= 1.10
    s = sum(w_mod[x] for x in keys)
    for x in keys: w_mod[x] /= s
    scores_mod = [calculate_score(row['composition_json'], w_mod)[0] for _, row in benchmark_df.iterrows()]
    rho, _ = spearmanr(base_scores, scores_mod)
    results[k] = rho
    print(f'Parameter {k:<18}: Spearman Rank Correlation = {rho:.4f}')

### 6. Visualizing Distribution & Robustness

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(results.keys()), list(results.values()), color='#16A34A')
ax.axvline(0.95, color='red', linestyle='--', label='Stability Threshold (0.95)')
ax.set_xlim(0.90, 1.0)
ax.set_title('Spearman Rank Correlation under +/-10% Weight Perturbation')
ax.set_xlabel('Rank Correlation Coefficient')
ax.legend()
plt.show()